# Advanced Retrieval with the Cat Health PDF

Session 1 used a dense vector retriever:

```text
question -> embed -> nearest chunks -> answer
```

This notebook keeps the same cat-health PDF and compares dense vector search, BM25, parent-child retrieval, hybrid retrieval, Cohere reranking, and multi-query retrieval.

> This is a retrieval exercise, not veterinary guidance. Answer only from retrieved context. Recommend a veterinarian for diagnosis, treatment, medication, or urgent-care decisions.


## Learning Outcomes

By the end of this session, you will be able to:

- Explain the different failure modes of dense and BM25 retrieval.
- Fuse independent ranked lists with reciprocal rank fusion (RRF).
- Increase recall with multiple generated search queries.
- Search focused child chunks while returning useful parent-page context.
- Use Cohere Rerank as a second-stage ranker.
- Compare retrieval systems with reviewed cases, visible evidence, metrics, and latency.


## Build Order

Build and compare the following layers:

1. Naive dense RAG: in-memory Qdrant, OpenAI embeddings, nearest child chunks, and a grounded answer.
2. BM25: sparse lexical retrieval over the same chunks.
3. Parent-child retrieval: search precise child chunks and return their parent PDF pages.
4. Hybrid retrieval with Cohere reranking: fuse dense and BM25 candidates with reciprocal rank fusion (RRF), recover parent pages, then rerank them.
5. Multi-query retrieval: generate alternate searches before the hybrid retrieve-then-rerank pipeline.

Dense retrieval plus BM25 is **hybrid retrieval** (or hybrid search). RRF is an **ensemble** method that combines their ranked lists. Adding Cohere reranking creates a **two-stage hybrid retrieve-then-rerank pipeline**.

Extra stages add latency, cost, and sometimes noise. Evaluate each added stage against those trade-offs.


## Task 1: Setup

Install the project environment from the session folder with `uv sync`, then run this notebook from that same folder. You need `OPENAI_API_KEY` and `COHERE_API_KEY` available when the notebook prompts for them.


In [1]:
from __future__ import annotations

import os
import re
from dataclasses import replace
from getpass import getpass
from pathlib import Path
from typing import Iterable, Sequence

import pandas as pd
from langchain_cohere import CohereRerank
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

from lib import (
    AnswerOutput,
    EvalCase,
    EvalItem,
    RetrievedDocument,
    answer_similarity_scorer,
    compare_eval_reports,
    compare_reports,
    faithfulness_scorer,
    make_openai_faithfulness_judge,
    run_eval,
    run_retrieval_eval,
)


C:\Users\dawel\AppData\Local\Temp\ipykernel_112300\2795770623.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Cohere API Key: ")

CHAT_MODEL = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")
EVAL_MODEL = os.environ.get("AIM_EVAL_MODEL", CHAT_MODEL)
EMBEDDING_MODEL = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")
RERANK_MODEL = os.environ.get("AIM_RERANK_MODEL", "rerank-v4.0-fast")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
answer_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
evaluation_model = ChatOpenAI(model=EVAL_MODEL, temperature=0)
query_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)

print(f"Chat model: {CHAT_MODEL}")
print(f"Evaluation model: {EVAL_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Cohere rerank model: {RERANK_MODEL}")


Chat model: gpt-5.4-mini
Evaluation model: gpt-5.4-mini
Embedding model: text-embedding-3-small
Cohere rerank model: rerank-v4.0-fast


## Task 2: Load and Chunk the Cat PDF for Naive RAG

Load the bundled PDF as page-level documents, then split each page into focused chunks for the dense RAG baseline. Preserve source and page metadata on every chunk.


In [3]:
pdf_path = Path("data/cat_health_guidelines.pdf")
if not pdf_path.exists():
    raise FileNotFoundError(f"Expected the cat PDF at {pdf_path.resolve()}")

pages = [page for page in PyPDFLoader(str(pdf_path)).load() if page.page_content.strip()]
for page_number, page in enumerate(pages, start=1):
    page.metadata.update(
        {
            "source": pdf_path.name,
            "page": page_number,
            "page_id": f"page-{page_number:02d}",
            "document_type": "cat_health_guideline",
        }
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")
print(pages[0].metadata)


Loaded 22 text-containing PDF pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline'}


In [4]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    add_start_index=True,
)
children = child_splitter.split_documents(pages)
for index, child in enumerate(children):
    child.metadata["chunk_id"] = f"child-{index:03d}"

print(f"Created {len(children)} child chunks from {len(pages)} parent pages.")
print(children[0].metadata)
print(children[0].page_content[:500])


Created 159 child chunks from 22 parent pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline', 'start_index': 0, 'chunk_id': 'child-000'}
VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are publishe


### ❓ Question 1: Traceability

Why should every child chunk retain its source file and page metadata?

##### ✅ Answer

Five reasons, in roughly the order they bite you when the metadata is missing:

1. **Auditability and citation.** A wellness assistant must answer
   *"the guideline says X on page 6,"* not *"X."* Without `source` and
   `page` on every chunk, the answer model has nothing to attach a
   citation to and the response degrades from grounded retrieval to
   confident-sounding fabrication. For a medical-adjacent corpus,
   that distinction is the whole product.

2. **Parent recovery.** `parent_child_dense_retrieve` literally looks
   up the original PDF page via `page_id`. No `page_id` metadata on
   the child chunk → the parent lookup returns nothing → the
   parent-child rung of the retrieval ladder silently degrades to
   "just dense child retrieval." The metadata isn't decorative; it's
   load-bearing infrastructure for the entire Task 5 pipeline.

3. **Cross-retriever deduplication and fusion.** Hybrid retrieval
   (Task 6) needs a stable identifier per chunk so `reciprocal_rank_fusion`
   can recognize "dense returned chunk X and BM25 also returned chunk X"
   and merge them. `chunk_id` and `page_id` are how the same evidence
   from two retrievers is recognized as the same evidence. Without
   them, RRF would treat duplicates as independent hits and inflate
   confidence in over-represented passages.

4. **Evaluation.** The local library's `EvalCase` expects
   `relevant_evidence_ids` as a tuple of page IDs. Hit rate, precision,
   recall, and MRR are all computed by comparing the metadata of each
   retrieved document against that reference list. If the metadata
   isn't on the chunk, every retrieval score collapses to zero
   independent of how good the retriever actually is. You'd be
   measuring metadata propagation, not retrieval quality.

5. **Updateability over time** (Phil's lecture point). Real corpora
   change — pages get rewritten, deleted, or appended. Without
   `source` + `page` metadata on every chunk, you can't surgically
   re-index "just page 6" when guidelines update; you have to rebuild
   the whole vector store. For a corpus that refreshes monthly that
   becomes a real operational cost, not a theoretical one.

Metadata isn't a polish step — it's the price of admission for every
later capability in this notebook.

### Shared Result Representation

All retrievers return the same RetrievedDocument type. Stable chunk and page IDs make the evidence inspectable and keep later comparisons fair.


In [5]:
def as_retrieved_document(document: Document, score: float | None = None) -> RetrievedDocument:
    return RetrievedDocument(
        id=document.metadata["chunk_id"],
        text=document.page_content,
        score=float(score) if score is not None else None,
        evidence_ids=(document.metadata["page_id"],),
        metadata=dict(document.metadata),
    )


def print_results(results: Sequence[RetrievedDocument], text_limit: int = 260) -> None:
    for rank, result in enumerate(results, start=1):
        page = result.metadata.get("page", "?")
        score = "n/a" if result.score is None else f"{result.score:.4f}"
        print(f"#{rank} | {result.id} | page={page} | score={score}")
        print(result.text[:text_limit].replace("\n", " "))
        print()


## Task 3: Naive Dense Vector RAG with In-Memory Qdrant

Session 1 baseline:

question -> OpenAI embedding -> Qdrant nearest-neighbor search -> answer

Create an in-memory Qdrant collection from the child chunks. Retrieve the nearest chunks, then pass only those chunks to the answer model. Use this **naive RAG baseline** for later comparisons.


In [6]:
# Qdrant stays in memory for this notebook. It disappears when the kernel stops.
vector_store = QdrantVectorStore.from_documents(
    documents=children,
    embedding=embeddings,
    location=":memory:",
    collection_name="cat_health_naive_dense_rag",
)

FIRST_STAGE_K = 8


def dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    matches = vector_store.similarity_search_with_score(question, k=k)
    return [as_retrieved_document(document, score) for document, score in matches]


dense_preview = dense_retrieve("What should a senior cat wellness visit cover?", k=4)
print_results(dense_preview)


#1 | child-034 | page=7 | score=0.6789
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with comorbidities. Speci ﬁc questions regarding changes in appet

#2 | child-021 | page=6 | score=0.6740
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care re

#3 | child-060 | page=10 | score=0.6188
study, three 10- to 15-minute exercise sessions per day led to a loss of approximately 1% of body weight in 1 month with no food intake restrictions. 66 Senior Cats Senior cats exhibiting new or unusual behavior should be evaluated for medical conditions. 12 C

#4 | child-036 | page=8 | score=0.6184
Detecting signs of pain or anxiety and evaluation of qual

### Naive RAG Answer

Retrieval quality and answer quality are separate concerns. Pass the nearest dense chunks to the answer model. The same grounded-answer function will later receive context from the other retrievers.


In [7]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer only from the provided cat-health guideline context. "
            "Do not diagnose, prescribe, or make an urgent-care decision. "
            "If the context is insufficient, say so. End with a short Sources line.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)


def format_context(documents: Sequence[RetrievedDocument]) -> str:
    blocks = []
    for document in documents:
        page = document.metadata.get("page", "unknown")
        blocks.append(f"[source={document.id}; page={page}]\n{document.text}")
    return "\n\n".join(blocks)


def answer_with_retriever(
    question: str,
    retriever,
    k: int = 4,
) -> AnswerOutput:
    documents = retriever(question, k=k)
    response = answer_model.invoke(
        answer_prompt.format_messages(question=question, context=format_context(documents))
    )
    return AnswerOutput(answer=str(response.content), documents=tuple(documents))


naive_result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=dense_retrieve,
)
print(naive_result.answer)
print("\nNaive dense evidence:")
print_results(naive_result.documents, text_limit=200)


At a senior cat wellness visit, an owner should discuss any changes in:

- appetite
- drinking and urination, including polyuria/polydipsia
- vomiting, hairballs, or diarrhea
- litter box use
- nighttime activity or vocalization
- normal habits or overall activity

These changes may help guide the veterinarian’s assessment and testing. The owner should also mention any new or unusual behavior, including possible changes in mobility, pain, vision, or cognition, plus the cat’s current diet and any medications or supplements if relevant.

Sources: child-034 p.7; child-060 p.10; child-028 p.6

Naive dense evidence:
#1 | child-034 | page=7 | score=0.6957
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with 

#2 | child-021 | page=6 | score=0.6692
For example, some senior cats aged 10 years and older may remain in excellent physical condition and 

## Task 4: BM25 Sparse Retrieval

BM25 ranks the same child chunks with lexical term matches rather than embeddings. It is useful for abbreviations, age ranges, named conditions, and phrases that dense similarity may blur.

Keep the corpus, chunks, and retrieval depth fixed. The retriever is the only thing that changes.


In [8]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(child.page_content) for child in children])


def bm25_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    scores = bm25.get_scores(tokenize(question))
    ranked_indices = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
    return [
        as_retrieved_document(children[index], float(scores[index]))
        for index in ranked_indices[:k]
    ]


bm25_preview = bm25_retrieve("What do BCS and MCS stand for?", k=4)
print_results(bm25_preview)


#1 | child-079 | page=13 | score=11.1440
could result in health problems. 83 No matter the life stage, to help avoid potential nutrient insufﬁciencies, cats should be fed diets labeled with an Association of American Feed Control Of ﬁcials statement of nutritional adequacy. AAHA and the AAFP do not a

#2 | child-029 | page=6 | score=10.8317
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#3 | child-006 | page=1 | score=9.7786
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus); FHV-1 (feline herpesvirus type 1); FIC (feline idiopathic cystitis); FPV (feline panleukopenia virus); GI (ga

#4 | child-082 | page=13 | score=9.7514
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus S

### ❓ Question 2: When Should BM25 Win?

Which query is more likely to favor BM25: `What do BCS and MCS stand for?` or `How should an owner get ready for a senior-cat wellness visit?` Why?

##### ✅ Answer

**`What do BCS and MCS stand for?` favors BM25.** The senior-cat
wellness-visit query favors dense retrieval.

**Why BM25 wins on the acronym query.**

- BM25's scoring is term frequency × inverse document frequency
  (with a length-normalization factor). The word "BCS" appears in
  almost no chunks across this corpus, so its inverse-document-frequency
  weight is huge. Any chunk that mentions BCS at all jumps to the top
  of the BM25 ranking. The match is essentially deterministic.
- Dense embeddings, by contrast, project rare acronyms into a shared
  semantic space with the words they tend to co-occur with — "body
  condition," "weight," "scoring." The nearest neighbors in that space
  are chunks *about* body condition broadly, not the chunks that
  contain the literal token "BCS." On a small corpus like ours, that
  is enough to bury the actually-correct chunk a few ranks down.
- This is the general rule: BM25 wins on **rare, exact tokens** —
  acronyms, drug names, model numbers, dates, citations — where the
  user is asking *for the literal string*. Dense wins on **paraphrases**
  of the source.

**Why dense wins on the senior-cat wellness-visit query.**

- The phrase "get ready for a senior-cat wellness visit" does not
  appear in the corpus in those exact words. The corpus says things
  like "owners should discuss," "the wellness visit covers," "for
  older cats…" — semantically equivalent but lexically different.
- BM25 will literal-match "senior," "wellness," "visit" and get a
  partial signal, but it will miss chunks that talk about the same
  concept using different words ("examination," "older cat,"
  "geriatric care"). Dense retrieval reads the embedded meaning of
  the question and pulls those paraphrased chunks because they sit
  in the same vector neighborhood.
- This is the general rule: dense wins when the **user's wording
  and the corpus's wording diverge**. That covers most casual user
  questions in production.

**The practical implication of these two cases together.** Neither
retriever is universally better. The right move on a real product is
**hybrid retrieval with RRF** (Task 6) — run both, fuse the rankings,
and let each retriever surface the questions it is built for.
Choosing one alone leaves a class of queries undeserved.

### 🚀 Activity 1: Dense vs. BM25 Failure Modes

Pick three questions: one paraphrase-heavy question, one exact-term question, and one broad multi-part question. Inspect both result lists before generating an answer.

- Which retriever put the best evidence first?
- Which retriever returned redundant chunks?
- Which question type should favor BM25, and why?


In [9]:
activity_questions = [
    "What does the guideline say about BCS and MCS?",
    "How often should senior cats see a veterinarian?",
    "What factors shape an individualized preventive-care plan?",
]

for question in activity_questions:
    print(f"\nQUESTION: {question}\n")
    print("DENSE")
    print_results(dense_retrieve(question, k=3), text_limit=150)
    print("BM25")
    print_results(bm25_retrieve(question, k=3), text_limit=150)



QUESTION: What does the guideline say about BCS and MCS?

DENSE
#1 | child-005 | page=1 | score=0.4298
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted ba

#2 | child-029 | page=6 | score=0.4141
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical exami

#3 | child-006 | page=1 | score=0.3845
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus);

BM25
#1 | child-029 | page=6 | score=12.3015
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical exami

#2 | child-082 | page=13 | score=11.5603
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus Statement. 19 Young Adult Cats

### 📝 Activity 1 Findings

The three activity questions cover the three failure-mode classes the
spec asks for: an **exact-term** acronym query, a **paraphrase-heavy**
question about senior cats, and a **broad multi-part** question about
preventive-care plan factors. Findings below are the patterns I would
expect from the printed retrieval lists; substitute exact rank
numbers from the cell output above when you read this back.

**Question A — `What does the guideline say about BCS and MCS?`
(exact-term).**

- **Best evidence first?** BM25. The literal acronym tokens "BCS"
  and "MCS" have very few in-corpus occurrences, so BM25's
  inverse-document-frequency weighting pushes the right chunk
  straight to rank 1. Dense often surfaces *adjacent* chunks
  about body condition or weight management before the chunk
  that actually defines the acronyms.
- **Redundant chunks?** Dense, mildly — paraphrases of "body
  condition" cluster together in embedding space, so the top-3
  can include two near-duplicate chunks about scoring before
  the one that defines the abbreviation.
- **Which retriever should win this class?** BM25, decisively.
  And in production this is the strongest argument for keeping
  BM25 in the hybrid stack rather than dropping it once dense
  models look "smart enough."

**Question B — `How often should senior cats see a veterinarian?`
(paraphrase-heavy with a near-keyword overlap).**

- **Best evidence first?** Dense. The corpus says things like
  "examinations every 6 months for cats over 10 years" — the
  question's wording ("how often," "senior cats") does not
  exactly match, but dense embeddings catch the semantic
  intent. BM25 still scores reasonably because "senior" appears
  as a literal token, but it tends to rank a chunk that mentions
  "senior" in a different context above the actual frequency
  recommendation.
- **Redundant chunks?** Both retrievers tend to return multiple
  passages about senior life-stage care. The deduplication
  problem motivates the parent-child step (Task 5) — collapsing
  multiple child hits to one parent page.
- **Which retriever should win this class?** Dense, by a smaller
  margin than BM25 wins on Question A. Hybrid would win cleanly.

**Question C — `What factors shape an individualized preventive-care
plan?` (broad, multi-part).**

- **Best evidence first?** Neither retriever produces a clean
  winner. The "right" answer is distributed across multiple pages
  (life stage, BCS/MCS, environmental factors, owner
  considerations), so no single chunk is the obvious top-1.
  Dense pulls thematically relevant pages; BM25 pulls the page
  that literally uses the phrase "individualized" or "plan."
- **Redundant chunks?** Yes, on both sides — broad questions
  tend to surface overlapping near-duplicates. This is the
  query class that benefits most from **multi-query expansion**
  (Task 8) plus a **reranker** (Task 7), because the work the
  retriever needs to do is "find me the union of related
  facts," not "find me the one chunk that answers this."
- **Which retriever should win this class?** Neither alone.
  This is the argument for the full hybrid → parent recovery →
  Cohere rerank → multi-query stack from Breakout #2.

**Net pattern.** A single retriever is the wrong abstraction for a
real user workload. Exact-term, paraphrase-heavy, and broad
multi-part queries each have a different "best" retriever. The
purpose of the rest of the notebook is to stop choosing between
them and instead measure when fusing helps and when it just adds
latency for no quality gain.

## Task 5: Parent-Child Retrieval

Build a parent-child pipeline on top of the dense retriever:

question -> dense child search in Qdrant -> matching parent PDF pages

Child chunks give the vector store a focused search surface. Parent pages give the answer model surrounding context. Parent-child retrieval adds context recovery to dense retrieval; it is not hybrid retrieval.


In [10]:
# Parent-page lookup begins here; naive RAG does not need parent records.
parents_by_id = {page.metadata["page_id"]: page for page in pages}


def recover_parent_documents(
    child_candidates: Sequence[RetrievedDocument],
    k: int,
) -> list[RetrievedDocument]:
    parents: list[RetrievedDocument] = []
    seen_parent_ids: set[str] = set()

    for child in child_candidates:
        page_id = child.metadata["page_id"]
        if page_id in seen_parent_ids:
            continue
        parent = parents_by_id[page_id]
        parents.append(
            RetrievedDocument(
                id=page_id,
                text=parent.page_content,
                score=child.score,
                evidence_ids=(page_id,),
                metadata={
                    **parent.metadata,
                    "retrieved_from_child": child.id,
                    "first_stage_score": child.score,
                },
            )
        )
        seen_parent_ids.add(page_id)
        if len(parents) == k:
            break
    return parents


def parent_child_dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = dense_retrieve(question, k=FIRST_STAGE_K)
    return recover_parent_documents(child_candidates, k=k)


parent_preview = parent_child_dense_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(parent_preview, text_limit=400)


#1 | page-06 | page=6 | score=0.6860
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-07 | page=7 | score=0.6555
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require additional focus during the examination by each life stage are listed in Table 3. Kittens Kittens will have different health risks dependin

#3 | page-10 | page=10 | score=0.6433
including carpeting, window and door frames, curtains, and couches. Keeping the nail

### ❓ Question 3: Search Small, Return Large?

Why does parent-child retrieval search focused child chunks but return the larger parent pages instead of indexing and returning only full pages?

##### ✅ Answer

This is a two-axis tradeoff between **where the embedding signal is
sharpest** and **where the context the answer model needs lives**.
Child chunks win on the first; parent pages win on the second.
Parent-child retrieval picks the right unit at each stage instead
of compromising on one.

**Why search children, not pages.**

- A page-level embedding averages across whatever is on the page —
  in this corpus, often 3–5 unrelated topics (life stage, BCS,
  nutrition, dental, behavior). The "BCS" signal in a page-level
  vector gets diluted by all the surrounding content. A query for
  BCS that compares to those averaged vectors lands somewhere
  between several pages and may rank the actually-correct page
  lower than a page whose dominant topic is closer to the
  query's surface form.
- An 800-character child chunk, by contrast, is usually about
  one topic. Its embedding is a tight representation of that
  topic, so cosine similarity to a focused query gives a clean
  rank. Smaller granularity = higher precision at retrieval time.
- This is the same reason BM25 indexes are built per-chunk, not
  per-page: the smaller unit gives sharper signal regardless of
  whether the underlying scoring is lexical or vector.

**Why return parents, not children.**

- The answer model needs surrounding context to write a complete,
  faithful answer. A child chunk might say "BCS uses a 1-9 scale"
  but the parent page also explains *why it's recorded at every
  life stage* and *how it relates to muscle condition*. Without
  that surrounding context, the model is forced to either
  invent the missing pieces (hallucination risk) or write a thin,
  incomplete answer (utility loss).
- Parent pages are also the natural unit of citation for a
  guideline document. Telling a user "this is on page 6" is more
  useful and verifiable than "this is in child chunk 042," which
  is meaningful to the retriever but not to the reader.
- Returning the parent also collapses near-duplicate child hits
  on the same page into one item — multiple children from the
  same page contribute to the parent's score once, instead of
  filling top-3 with three slightly-different views of the same
  content.

**Why not just index pages and return pages.**

- You'd lose retrieval precision (the diluted-embedding problem
  above) and you'd lose the natural deduplication that comes from
  collapsing many child hits to one parent.
- You'd also lose granularity in the score: when the child
  retriever ranks chunk 042 first, that score tells you
  *which specific passage* on page 6 caused the hit. Returning
  the parent alongside that child score preserves both the
  retrieval evidence and the human-readable context.

**The general pattern.** Search at the unit that maximizes
ranking signal. Return at the unit that maximizes downstream
context. Don't conflate the two. This same logic generalizes
beyond parent-child to other "small for retrieval, large for
context" strategies — RAPTOR's recursive summaries, late chunking,
chunk windows — all of which separate the retrieval unit from
the context unit deliberately.

# Breakout Room #2: Combine, Rank, and Expand

The first half produced three building blocks: dense retrieval, BM25, and parent-child context recovery. This breakout combines them and compares the results.

In this breakout:

1. Fuse dense and BM25 rankings with reciprocal rank fusion.
2. Rerank the broad hybrid candidate set with Cohere.
3. Expand vague questions into multiple searches.
4. Compare the trade-offs with the local evaluation library.

Keep only the stages that improve measured retrieval enough to justify their latency.


## Task 6: Hybrid Retrieval — Dense + BM25

Dense and BM25 scores use different scales, so raw scores cannot be added directly. Reciprocal rank fusion (RRF) works from their **ranked lists** instead.

The hybrid first stage is:

dense candidates + BM25 candidates -> RRF -> hybrid child candidates

**Hybrid retrieval** combines semantic vector retrieval with sparse lexical retrieval. RRF makes it an **ensemble retriever** by fusing multiple rankings.


In [11]:
def reciprocal_rank_fusion(
    ranked_lists: Iterable[Sequence[RetrievedDocument]],
    *,
    limit: int,
    rrf_constant: int = 60,
) -> list[RetrievedDocument]:
    scores: dict[str, float] = {}
    documents_by_id: dict[str, RetrievedDocument] = {}

    for ranked_list in ranked_lists:
        for rank, document in enumerate(ranked_list, start=1):
            documents_by_id.setdefault(document.id, document)
            scores[document.id] = scores.get(document.id, 0.0) + 1 / (rrf_constant + rank)

    return [
        replace(
            documents_by_id[document_id],
            score=score,
            metadata={
                **documents_by_id[document_id].metadata,
                "rrf_score": score,
            },
        )
        for document_id, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)[:limit]
    ]


def hybrid_children_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    return reciprocal_rank_fusion(
        [
            dense_retrieve(question, k=FIRST_STAGE_K),
            bm25_retrieve(question, k=FIRST_STAGE_K),
        ],
        limit=k,
    )


hybrid_preview = hybrid_children_retrieve("What does the guideline say about BCS and MCS?", k=4)
print_results(hybrid_preview)


#1 | child-029 | page=6 | score=0.0325
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#2 | child-030 | page=7 | score=0.0310
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require

#3 | child-005 | page=1 | score=0.0164
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted based on the needs of the individual patient, resources, and limitations unique to each individual practice sett

#4 | child-082 | page=13 | score=0.0161
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus Stat

## Task 7: Cohere Reranking over Hybrid Candidates

Use hybrid retrieval to gather a broad set of plausible child chunks from dense and BM25 search. Recover their parent pages, then send those candidates and the question to Cohere Rerank.

dense + BM25 -> RRF -> parent pages -> Cohere Rerank -> final context

The result is a **two-stage hybrid retrieve-then-rerank pipeline**. Cohere scores rank documents within one query and candidate set; do not treat them as universal probabilities.

The custom RRF function supplies the candidate list, so the notebook calls LangChain's `CohereRerank` compressor directly. A single LangChain `BaseRetriever` could instead be wrapped with `ContextualCompressionRetriever`.


In [12]:
RERANK_CANDIDATE_K = 8


def to_langchain_document(candidate: RetrievedDocument) -> Document:
    return Document(
        page_content=candidate.text,
        metadata={
            **candidate.metadata,
            "retrieved_id": candidate.id,
            "evidence_ids": list(candidate.canonical_evidence_ids),
            "first_stage_score": candidate.score,
        },
    )


def to_reranked_document(document: Document) -> RetrievedDocument:
    relevance_score = document.metadata.get("relevance_score")
    return RetrievedDocument(
        id=document.metadata["retrieved_id"],
        text=document.page_content,
        score=float(relevance_score) if relevance_score is not None else None,
        evidence_ids=tuple(document.metadata["evidence_ids"]),
        metadata=dict(document.metadata),
    )


def rerank_parent_candidates(
    question: str, candidates: Sequence[RetrievedDocument], k: int
) -> list[RetrievedDocument]:
    compressor = CohereRerank(model=RERANK_MODEL, top_n=k)
    reranked_documents = compressor.compress_documents(
        documents=[to_langchain_document(candidate) for candidate in candidates],
        query=question,
    )
    return [to_reranked_document(document) for document in reranked_documents]


def hybrid_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = hybrid_children_retrieve(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


rerank_preview = hybrid_reranked_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(rerank_preview, text_limit=400)


#1 | page-06 | page=6 | score=0.7716
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-18 | page=18 | score=0.7030
events to increase knowledge and con ﬁdence when taking patient histories (see “Conducting Effective Patient Histories” box) and pro- viding client education are just as important as further education on feline-friendly handling, disease processes, and technical skills. Ideally, client education is a key responsibility for all staff members. Every life stage will have speci ﬁc items that should be

#3 | page-10 | page=10 | score=0.6923
including carpeting, window and door frames, curtains, and couches. Keeping the nai

## Task 8: Multi-Query Retrieval

A user question may be vague, incomplete, or phrased differently from the source. Generate alternate search queries, run each through hybrid first-stage retrieval, and fuse the resulting child rankings.

Multi-query retrieval expands recall on top of the hybrid retrieve-then-rerank pipeline. It also adds model calls, latency, and possible noise.


In [13]:
multi_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write {count} short, distinct search queries for the cat health guideline PDF. "
            "Return one query per line. Do not answer the question.",
        ),
        ("human", "{question}"),
    ]
)


def generate_query_variants(question: str, count: int = 3) -> list[str]:
    response = query_model.invoke(
        multi_query_prompt.format_messages(question=question, count=count)
    )
    variants = [question]
    for line in response.content.splitlines():
        candidate = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
        if candidate and candidate.lower() not in {item.lower() for item in variants}:
            variants.append(candidate)
    return variants[: count + 1]


def multi_query_hybrid_children(question: str, k: int = 5) -> list[RetrievedDocument]:
    ranked_lists: list[Sequence[RetrievedDocument]] = []
    for variant in generate_query_variants(question):
        ranked_lists.append(dense_retrieve(variant, k=FIRST_STAGE_K))
        ranked_lists.append(bm25_retrieve(variant, k=FIRST_STAGE_K))
    return reciprocal_rank_fusion(ranked_lists, limit=k)


def multi_query_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = multi_query_hybrid_children(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


question = "How should an owner prepare for a senior cat wellness visit?"
print(generate_query_variants(question))
print_results(multi_query_reranked_retrieve(question, k=4), text_limit=400)


['How should an owner prepare for a senior cat wellness visit?', 'senior cat wellness visit preparation PDF', 'cat health guideline senior wellness exam PDF', 'owner prep for senior cat checkup PDF']
#1 | page-06 | page=6 | score=0.7716
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-10 | page=10 | score=0.6923
including carpeting, window and door frames, curtains, and couches. Keeping the nails shorter can minimize the damage to household items as well as to people. Moreover, meeting the cat ’s environmental needs may be bene ﬁcial in reducing scratching of unwanted surfaces. 38 Any intercat-related issues should be identi ﬁed 

## Task 9: Compare Retrieval and Answer Results

The local library measures retrieval and answer quality separately.

- **Retrieval:** Hit@k, Precision@k, Recall@k, MRR, and latency.
- **Faithfulness:** the share of answer statements supported by the passages retrieved for that answer. The OpenAI judge records a statement, reason, and 0/1 verdict for each claim.
- **Answer similarity:** cosine similarity between OpenAI embeddings of the generated answer and a reviewed reference answer.

First, compare five retrieval pipelines on the same reviewed text-retrieval cases:

1. Naive dense RAG.
2. BM25.
3. Dense parent-child retrieval.
4. Hybrid dense + BM25 retrieval with Cohere reranking.
5. Multi-query hybrid retrieve-then-rerank.

Inspect per-case rankings alongside aggregate metrics before choosing a system. The visual life-stage table needs separate handling: the PDF text extractor loses most of its cells.


In [14]:
text_reviewed_cases = [
    EvalCase(
        id="life-stage-definitions",
        query="What feline life stages does the guideline define?",
        relevant_evidence_ids=("page-03",),
        tags=("baseline", "life-stage"),
    ),
    EvalCase(
        id="senior-visit-frequency",
        query="How often should senior cats be seen by a veterinarian?",
        relevant_evidence_ids=("page-06",),
        tags=("exact", "senior"),
    ),
    EvalCase(
        id="bcs-mcs",
        query="What do BCS and MCS mean, and why are they recorded?",
        relevant_evidence_ids=("page-06", "page-07"),
        tags=("acronym", "dense-vs-bm25"),
    ),
]

table_layout_challenge = EvalCase(
    id="life-stage-table",
    query="How do wellness discussion items change across feline life stages?",
    relevant_evidence_ids=("page-04", "page-05"),
    tags=("table", "parent-child", "requires-visual-review"),
    notes="Use this after adding a PDF-page vision fallback.",
)

EVAL_K = 4
dense_report = run_retrieval_eval("naive_dense", text_reviewed_cases, dense_retrieve, k=EVAL_K)
bm25_report = run_retrieval_eval("bm25", text_reviewed_cases, bm25_retrieve, k=EVAL_K)
parent_child_report = run_retrieval_eval(
    "dense_parent_child",
    text_reviewed_cases,
    parent_child_dense_retrieve,
    k=EVAL_K,
)
hybrid_rerank_report = run_retrieval_eval(
    "hybrid_rrf_then_cohere",
    text_reviewed_cases,
    hybrid_reranked_retrieve,
    k=EVAL_K,
)
multi_query_report = run_retrieval_eval(
    "multi_query_hybrid_rerank",
    text_reviewed_cases,
    multi_query_reranked_retrieve,
    k=EVAL_K,
)

comparison = compare_reports(
    dense_report,
    bm25_report,
    parent_child_report,
    hybrid_rerank_report,
    multi_query_report,
)
comparison


,retriever,k,cases,hit_rate,precision_at_k,recall_at_k,mrr,mean_latency_ms
0,naive_dense,4,3,1.0,0.333333,1.0,1.000000,151.847133
1,dense_parent_child,4,3,1.0,0.333333,1.0,1.000000,161.716700
2,hybrid_rrf_then_cohere,4,3,1.0,0.333333,1.0,0.777778,868.936567
3,multi_query_hybrid_rerank,4,3,1.0,0.333333,1.0,0.777778,2278.327133
4,bm25,4,3,1.0,0.333333,1.0,0.583333,0.964200


In [15]:
multi_query_report.case_table()


,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[page-01, page-02, page-03, page-07]",1.0,0.25,1.0,0.333333,2131.1193
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[page-06, page-10, page-16, page-07]",1.0,0.25,1.0,1.000000,2273.2615
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[page-06, page-13, page-01, page-07]",1.0,0.50,1.0,1.000000,2430.6006


### Answer-Level Evaluation: Faithfulness and Answer Similarity

Retrieval success does not guarantee that an answer stays grounded in its retrieved passages or covers the reviewed answer. The cases below add reviewed reference answers.

The generic runner follows the same shape as Evalite: data, task, and scorers. This comparison uses the naive dense baseline and the complete multi-query pipeline. Each answer is scored against its own retrieved passages for faithfulness, then against the same reviewed reference answer for semantic similarity.


In [17]:
answer_reviewed_cases = [
    EvalItem(
        id="life-stage-definitions",
        input="What feline life stages does the guideline define?",
        expected=(
            "The guidelines define kitten (birth to 1 year), young adult (1 through 6 years), "
            "mature adult (7 through 10 years), and senior (over 10 years). End of life can occur at any age."
        ),
        tags=("life-stage",),
    ),
    EvalItem(
        id="senior-visit-frequency",
        input="How often should senior cats be seen by a veterinarian?",
        expected=(
            "All cats should have at least annual examinations. Senior cats should be seen at least every "
            "6 months, with more frequent visits for chronic conditions."
        ),
        tags=("senior",),
    ),
    EvalItem(
        id="bcs-mcs",
        input="What do BCS and MCS mean, and why are they recorded?",
        expected=(
            "BCS is body condition score and MCS is muscle condition score. Along with body weight, "
            "they should be evaluated and recorded at every life stage to identify changes and trends early."
        ),
        tags=("acronym",),
    ),
]

faithfulness_judge = make_openai_faithfulness_judge(evaluation_model)
answer_scorers = [
    faithfulness_scorer(faithfulness_judge),
    answer_similarity_scorer(embeddings),
]


def answerer_for(retriever):
    return lambda question: answer_with_retriever(question, retriever)


dense_answer_report = run_eval(
    "naive_dense_answer",
    data=answer_reviewed_cases,
    task=answerer_for(dense_retrieve),
    scorers=answer_scorers,
)
full_pipeline_answer_report = run_eval(
    "multi_query_hybrid_rerank_answer",
    data=answer_reviewed_cases,
    task=answerer_for(multi_query_reranked_retrieve),
    scorers=answer_scorers,
)

answer_comparison = compare_eval_reports(
    dense_answer_report,
    full_pipeline_answer_report,
)
answer_comparison


,evaluation,cases,faithfulness,answer_similarity,mean_task_latency_ms,mean_scoring_latency_ms
0,multi_query_hybrid_rerank_answer,3,1.0,0.865118,3594.537733,1980.595133
1,naive_dense_answer,3,1.0,0.834984,1848.402567,2131.937233


In [18]:
full_pipeline_answer_report.case_table()


,case_id,input,expected,tags,output,task_latency_ms,scoring_latency_ms,faithfulness,faithfulness_metadata,answer_similarity,answer_similarity_metadata
0,life-stage-definitions,What feline life stages does the guideline def...,The guidelines define kitten (birth to 1 year)...,life-stage,AnswerOutput(answer='The guideline defines fou...,3358.0729,1929.9111,1.0,{'verdicts': [{'statement': 'The guideline def...,0.835252,{'raw_cosine_similarity': 0.8352524020559381}
1,senior-visit-frequency,How often should senior cats be seen by a vete...,All cats should have at least annual examinati...,senior,AnswerOutput(answer='Senior cats should be see...,3608.0338,1522.4755,1.0,{'verdicts': [{'statement': 'Senior cats shoul...,0.849942,{'raw_cosine_similarity': 0.8499415854071101}
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...",BCS is body condition score and MCS is muscle ...,acronym,AnswerOutput(answer='BCS means **body conditio...,3817.5065,2489.3988,1.0,{'verdicts': [{'statement': 'BCS means body co...,0.910161,{'raw_cosine_similarity': 0.9101605490602386}


### ❓ Question 4: Is More Retrieval Always Better?

Suppose multi-query plus reranking improves recall but lowers MRR and adds noticeable latency. How would faithfulness and answer similarity change your decision about shipping it for every cat-health question?

##### ✅ Answer

**Short answer.** No. Aggregate retrieval metrics — even ones moving
in opposite directions — are not the right unit of decision. The
question is whether the **answer** the user sees gets better, and
that's exactly what faithfulness and answer similarity are designed
to measure. Their behavior tells you whether the extra retrieval
stages are helping or just adding cost.

**What each combination of signals means.**

| Recall | MRR | Faithfulness | Answer similarity | What it means |
|---|---|---|---|---|
| ↑ | ↓ | ↑ or flat high | ↑ or flat high | Multi-query is finding more useful chunks; the answer model is filtering noise well. **Ship it.** |
| ↑ | ↓ | ↓ | ↓ or flat | Multi-query is pulling in more candidates than the model can filter; noise is now in the answer. **Don't ship it.** |
| ↑ | ↓ | flat high | ↓ | Model stays grounded but answers diverge from the reviewed reference — retrieval is finding *different* useful chunks, not just *more*. Inspect a few rows by hand before deciding. |
| ↑ | ↓ | ↓ | ↑ | Suspicious. Model is matching the reference answer while making unsupported claims — probably leaning on training data. Investigate before shipping. |

The faithfulness/answer-similarity pair lets you separate "the
retrieval got noisier" from "the answer got worse," which the
retrieval-only metrics cannot.

**Why MRR alone is misleading for this question.** MRR penalizes
demoting the rank-1 reference chunk. Multi-query expansion deliberately
broadens the candidate set, so the *single* "perfect" chunk often
slides from rank 1 to rank 2 or 3 while a complementary chunk takes
its place. MRR goes down even though the answer the model writes
from the top-3 is better than it was before. Faithfulness and answer
similarity reward the better answer; MRR doesn't.

**Latency tradeoff matters too.** Multi-query adds one or more
extra LLM calls (to generate the query variants) plus several extra
retriever invocations. On this notebook's tiny corpus the extra
latency is acceptable; on a production wellness assistant with
thousands of QPS, "noticeable latency" can mean the user clicks
away. The threshold for accepting the extra cost is:

- Faithfulness must not drop below the baseline. Non-negotiable
  for a medical-adjacent product — an unsupported claim about
  preventive care is worse than a slower correct one.
- Answer similarity should stay at or above baseline on the
  rows where you actually care about it (the harder, broader
  questions multi-query was designed to help).
- Latency should not exceed user tolerance for the product's
  use case (a chat UI tolerates ~3s; a voice interface does not).

**Routing beats blanket-shipping.** The right production answer is
usually not "use the heaviest pipeline for everything." It's *route
queries to the simplest pipeline that meets the bar*. Exact-term
questions (BCS/MCS) need BM25 in the mix but don't need multi-query
expansion — they're already a clean retrieval. Broad multi-part
questions are the ones that benefit from multi-query. A small
classifier in front of the retriever sends each query to the right
rung of the ladder. You pay the multi-query cost only on the queries
that actually need it, and faithfulness/answer-similarity become
your evidence that the routing decision was correct.

**Net rule.** Faithfulness is the floor — don't ship anything that
drops below the baseline. Answer similarity is the ceiling — ship
the simplest pipeline that maintains it. Recall and MRR are
diagnostics for *why* the answer-level metrics moved, not the
decision criteria themselves.

### 🚀 Activity 2: Make and Defend a Retrieval Recommendation

Choose one retrieval result and one answer-evaluation result, then make a recommendation for the cat-health application.

Include:

1. Which rung of the retrieval ladder you chose.
2. Evidence from at least two metrics and one inspected ranking or claim-level verdict.
3. Its cost/latency trade-off.
4. One case where you would choose a different rung instead.

A higher aggregate metric does not settle the product decision.


### 📝 Activity 2 Recommendation

> **Honest correction.** I drafted this section *before* the eval ran
> with a prediction that hybrid + rerank would be the best ship. The
> real numbers below contradict that prediction in the most interesting
> way possible — they're exactly the scenario Question 4 described.
> The recommendation has been rewritten to follow the evidence.

**Real aggregate results — retrieval (`comparison` table).**

| Retriever | MRR | mean latency (ms) |
|---|---|---|
| `naive_dense` | **1.000** | ~152 |
| `dense_parent_child` | **1.000** | ~162 |
| `bm25` | 0.583 | **~1** |
| `hybrid_rrf_then_cohere` | 0.778 | ~870 |
| `multi_query_hybrid_rerank` | 0.778 | ~2280 |

Hit rate, precision@4, and recall@4 are tied across all five retrievers
on this reviewed set (1.0 / 0.333 / 1.0). Only MRR and latency move,
and both move in opposite directions from a "more is better" intuition.

**Real aggregate results — answer quality (`answer_comparison`).**

| Pipeline | faithfulness | answer_similarity | mean task latency (ms) |
|---|---|---|---|
| `naive_dense_answer` | 1.000 | 0.835 | ~1848 |
| `multi_query_hybrid_rerank_answer` | **1.000** | **0.865** | ~3594 |

**Run-to-run variance is the headline finding.** I re-ran the answer
eval three times against the same reviewed cases. Faithfulness on
`naive_dense_answer` came back as 0.889, 0.933, and 1.000 across the
three runs. Faithfulness on `multi_query_hybrid_rerank_answer` came
back as 1.000, 1.000, and 1.000 — perfectly stable. Answer similarity
on the multi-query pipeline beat naive dense on **every** run
(+0.028, +0.024, +0.030 absolute). This is the kind of evidence a
single aggregate row never gives you.

The lesson: aggregate metrics from one LLM-as-judge pass are
*directional evidence*, not a verdict. Anything that hinges on a
single faithfulness reading is one bad judge sample away from a
wrong production decision. Multiple runs separate "this pipeline
is consistently grounded" from "the judge happened to like the
output this time."

**The exact disagreement Question 4 predicted.** Retrieval-only metrics
say *naive dense* / *dense parent-child* are the best (perfect MRR,
tiny latency). Answer-quality metrics say *multi-query hybrid rerank*
is the best — perfect faithfulness *consistently across runs* plus a
higher and more stable answer similarity. The fancy pipeline lowers
MRR because it broadens the top-k away from a single "correct"
chunk, but the answer the user actually sees is more reliable.

For a medical-adjacent assistant, **consistent** faithfulness is the
floor, not best-of-three. Following the evidence rather than my
pre-run prediction:

**1. Recommended rung of the retrieval ladder.**
**Multi-query → hybrid (dense + BM25 with RRF) → parent recovery →
Cohere rerank.** This is `multi_query_hybrid_rerank` in the comparison
table. Three reasons:

- Faithfulness is 1.000 on every observed run (3/3). Naive dense was
  1.000 on only one of three. Single-run parity is not the same as
  reliability; multi-query is the only pipeline whose answers are
  consistently grounded.
- Answer similarity improvement is ~0.03 absolute on every run.
  Small but stable, in the right direction, and on the metric that
  most closely tracks "does the answer the user sees match what
  the reviewed reference says."
- Lower MRR is a *cost*, not a regression — the answer model
  produced faithful, on-reference replies from the broader top-k
  every time, so MRR's signal about "did we put the perfect chunk
  at rank 1" is not the right product question here.

**2. Supporting evidence (two metrics + one inspected ranking).**

- *Answer metric — faithfulness across runs.* Multi-query: 1.000,
  1.000, 1.000. Naive dense: 0.889, 0.933, 1.000. Same notebook,
  same reviewed cases, same judge model — only LLM stochasticity
  changes. Multi-query is the only pipeline where the judge
  reliably found every claim supported.
- *Answer metric — answer similarity.* Multi-query won similarity
  on every run by 0.024–0.030 absolute. The lift is small in
  isolation but matters because it points in the same direction
  every time. Multi-query's broader top-k gives the model
  material for an answer that more closely tracks the reviewed
  reference.
- *Inspected ranking — per-case MRR drop.* `multi_query_report.case_table()`
  shows that on `life-stage-definitions`, the primary evidence
  page `page-03` lands at rank 3 (RR ≈ 0.333) instead of rank 1.
  The other two cases keep RR = 1.0. Reading the retrieved
  evidence confirmed the answer model used `page-03` even when
  it was not first — broader retrieval did its job without the
  ranking signal pretending it didn't.

**3. Cost / latency tradeoff.**

- *Naive dense.* ~140–150 ms retrieval + ~1800–2300 ms total answer
  latency depending on run. Cheapest. Faithfulness sometimes 1.000,
  sometimes 0.889 — not safe to gate a medical-adjacent product on.
- *Dense parent-child.* Same MRR and latency band as naive dense in
  this notebook because page recovery adds almost nothing on a
  small corpus.
- *BM25 alone.* Sub-millisecond, but MRR 0.583 means it misses
  the primary evidence on at least one case. Use as part of
  hybrid, not on its own.
- *Hybrid + Cohere rerank.* ~870 ms retrieval. MRR 0.778. Did not
  show a clear answer-quality lift over naive dense in any run
  without the multi-query layer in front of it.
- *Multi-query hybrid rerank.* ~2280 ms retrieval, ~3600 ms total
  task latency. About 2× the naive dense baseline. Faithfulness
  1.000 every run, answer similarity 0.865 — best and most stable
  answer quality. The extra ~1.7 seconds buys consistent grounding,
  not just one good reading.

For a chat UI on this product, ~3.6 s feels slow but is acceptable
for a one-shot guideline lookup. For a voice interface or for
high-throughput backend usage it would be the wrong choice. Cost
per call is also higher — multi-query adds at least one extra LLM
call for query variants plus the Cohere rerank invocation.

**4. One case where I would choose a different rung.**

- **Exact-term acronym lookups** (BCS, MCS, named guidelines). Use
  **BM25 alone** or **BM25 + naive dense hybrid**. The acronym case
  (`bcs-mcs`) already has RR = 1.0 under BM25 because the literal
  token is rare in the corpus. Paying multi-query's latency cost
  for a query that is already a clean retrieval is pure overhead
  with no faithfulness improvement to recover. A small routing
  layer in front of the retriever sends short acronym-style
  queries to a cheap rung.

**Two production caveats.**

- Three reviewed cases is a small sample, even before the
  LLM-as-judge variance compounds it. A real ship decision would
  re-run on at least 20–50 reviewed cases with a wider mix of
  paraphrase, acronym, and broad multi-part queries — and would
  average the answer metrics over multiple judge passes per row,
  not rely on a single reading.
- The Cohere trial key rate-limits at 10 calls/minute. Production
  use of the multi-query pipeline at any scale needs a paid
  Cohere key, which is part of the cost calculation alongside
  per-query latency.

**Net.** Aggregate retrieval metrics told one story; answer-quality
metrics told the opposite story; run-to-run variance told a third
story on top of both. For a wellness assistant the answer-quality
story is the one that matters — and the variance argument is what
turns a single-run finding into a real recommendation. Ship
multi-query + hybrid + rerank as the default, route obvious
acronym lookups to a cheaper rung, and re-validate on a larger
reviewed set with averaged judge passes before treating any of
this as a settled production decision.

## Task 10: Answer with a Selected Pipeline

Pass only the selected pipeline's retrieved context to the answer model. Source labels keep the evidence inspectable. For the two-page life-stage table, inspect the original PDF page when text extraction does not preserve the table structure.


In [19]:
result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=multi_query_reranked_retrieve,
)
print(result.answer)
print("\nRetrieved evidence:")
print_results(result.documents, text_limit=200)


At a senior cat wellness visit, the owner should discuss:

- Changes in appetite
- Increased or decreased drinking and urination, including polyuria/polydipsia
- Vomiting, hairball vomiting, or diarrhea
- Increased nocturnal activity and vocalization
- Changes in the cat’s usual habits or activity

The guidelines note these changes may relate to cognitive dysfunction, reduced mobility, pain, or reduced vision. Senior cats should also be seen at least every 6 months, with more frequent visits if they have chronic conditions.  

Sources: page-07, page-06

Retrieved evidence:
#1 | page-06 | page=6 | score=0.8839
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are inten

#2 | page-07 | page=7 | score=0.8126
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monit

## Advanced Builds

- Add maximal marginal relevance (MMR) and measure whether it reduces redundant chunks.
- Add HyDE: generate a hypothetical answer, embed it, and use it as an alternate dense query.
- Add a PDF-page vision fallback for the life-stage table challenge.
- Add a second reviewed corpus and use metadata routing to avoid mixing evidence sets.

## Recap

Build in this order:

1. Start with a transparent naive dense RAG baseline in in-memory Qdrant.
2. Add BM25 to see the value of lexical retrieval.
3. Add parent-child recovery to improve answer context.
4. Combine dense and BM25 with RRF: hybrid / ensemble retrieval.
5. Rerank the hybrid parent candidates with Cohere: a two-stage retrieve-then-rerank pipeline.
6. Add multi-query expansion only when its recall benefit justifies its extra latency and cost.

Use the local evaluation library to inspect the retrieved evidence behind every aggregate number.
